# UIT_SGS_Foodseekers_C1 — **ENSEMBLE 3 MÔ HÌNH** · Lượt 5 (bản chủ lực)
Mục tiêu: điểm cao + **vững trên Private LB** để nhắm giải cao nhất (vượt 0.95082).

**Ý tưởng:** 3 mô hình khác bản chất, sai ở chỗ khác nhau nên bù lỗi cho nhau khi trộn xác suất:
1. `FacebookAI/xlm-roberta-large` — đa ngữ mạnh nhất (pt/fr/it/en).
2. `FacebookAI/xlm-roberta-base` — đa ngữ, nhẹ, ổn định.
3. `protectai/deberta-v3-base-prompt-injection-v2` — đã học sẵn về prompt injection (cực hợp nhãn hiếm PROMPT_INJECTION).

Tất cả đều nằm trong **danh sách Approved Models** của ban tổ chức.

**Chạy:** Runtime → T4 GPU → Run all. Khá lâu (~90–120 phút, 3 model × 3 fold). Đừng để máy sleep.
Mỗi model train xong tự lưu xác suất ra file `.npy`, nên nếu rớt phiên vẫn giữ lại được phần đã chạy.

In [ ]:
!pip -q install -U transformers accelerate sentencepiece
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO GPU - bat T4!")

In [ ]:
# Cấu hình chung
import random, numpy as np, pandas as pd
BASE_URL = "https://raw.githubusercontent.com/bersious/Foodseekers/main/data"
LABELS = ["BENIGN","JAILBREAK","PROMPT_INJECTION"]; L2I={l:i for i,l in enumerate(LABELS)}
SEED, N_FOLDS = 42, 3
# (model_name, max_len, batch, epochs, lr, trọng số khi ensemble)
MODELS = [
    ("FacebookAI/xlm-roberta-large", 256, 8,  3, 1.5e-5, 0.45),
    ("FacebookAI/xlm-roberta-base",  320, 16, 4, 2e-5,   0.30),
    ("protectai/deberta-v3-base-prompt-injection-v2", 320, 16, 4, 2e-5, 0.25),
]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
train = pd.read_csv(f"{BASE_URL}/train.csv"); test = pd.read_csv(f"{BASE_URL}/test.csv")
train["text"]=train["text"].fillna("").astype(str); test["text"]=test["text"].fillna("").astype(str)
y_all = train["label"].map(L2I).values
counts = np.bincount(y_all, minlength=3)
class_w = torch.tensor(len(y_all)/(3*counts), dtype=torch.float)
print(train.shape, test.shape, "| class weights:", class_w.numpy().round(3))

In [ ]:
import torch.nn as nn, copy
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.special import softmax

class DS(Dataset):
    def __init__(self, texts, labels=None): self.t=list(texts); self.l=labels
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return (self.t[i], -1 if self.l is None else int(self.l[i]))

class MSDHead(nn.Module):
    """Backbone bất kỳ + đầu phân loại Multi-Sample Dropout 3 lớp."""
    def __init__(self, name, n_cls=3, n_drop=5):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        h = self.backbone.config.hidden_size
        self.drops = nn.ModuleList([nn.Dropout(0.3) for _ in range(n_drop)])
        self.fc = nn.Linear(h, n_cls)
    def forward(self, input_ids=None, attention_mask=None, **kw):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]
        return sum(self.fc(d(pooled)) for d in self.drops) / len(self.drops)

def train_one_model(name, max_len, batch, epochs, lr):
    tok = AutoTokenizer.from_pretrained(name)
    def collate(b):
        t,l = zip(*b)
        enc = tok(list(t), truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(l); return enc

    oof = np.zeros((len(train),3)); test_prob = np.zeros((len(test),3)); fold_f1=[]
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k,(tr,va) in enumerate(skf.split(train, y_all)):
        print(f"    fold {k+1}/{N_FOLDS}")
        dl_tr = DataLoader(DS(train['text'].iloc[tr], y_all[tr]), batch_size=batch, shuffle=True, collate_fn=collate)
        dl_va = DataLoader(DS(train['text'].iloc[va], y_all[va]), batch_size=batch*2, collate_fn=collate)
        dl_te = DataLoader(DS(test['text']), batch_size=batch*2, collate_fn=collate)
        model = MSDHead(name).to(DEVICE)
        loss_fn = nn.CrossEntropyLoss(weight=class_w.to(DEVICE))
        optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        steps = len(dl_tr)*epochs
        sched = get_linear_schedule_with_warmup(optim, int(0.1*steps), steps)
        scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

        @torch.no_grad()
        def infer(dl):
            model.eval(); o=[]
            for enc in dl:
                enc.pop("labels"); enc={k:v.to(DEVICE) for k,v in enc.items()}
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    o.append(model(**enc).float().cpu().numpy())
            return np.concatenate(o)

        best, best_state = -1, None
        for ep in range(epochs):
            model.train()
            for enc in dl_tr:
                lb = enc.pop("labels").to(DEVICE); enc={k:v.to(DEVICE) for k,v in enc.items()}
                optim.zero_grad()
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    loss = loss_fn(model(**enc), lb)
                scaler.scale(loss).backward(); scaler.step(optim); scaler.update(); sched.step()
            f = f1_score(y_all[va], infer(dl_va).argmax(1), average="macro")
            if f>best: best, best_state = f, copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        oof[va] = softmax(infer(dl_va),1); test_prob += softmax(infer(dl_te),1)/N_FOLDS; fold_f1.append(best)
        del model; torch.cuda.empty_cache()
    print(f"  -> {name}: fold F1 {[round(x,4) for x in fold_f1]} | OOF {f1_score(y_all, oof.argmax(1), average='macro'):.4f}")
    return oof, test_prob

In [ ]:
# Train lần lượt 3 mô hình, lưu xác suất ra file (an toàn nếu rớt phiên)
oofs, tests, weights = [], [], []
for name, ml, bs, ep, lr, w in MODELS:
    print(f"\n========== {name} (w={w}) ==========")
    oof, tp = train_one_model(name, ml, bs, ep, lr)
    tag = name.split('/')[-1].replace('.','_')
    np.save(f"oof_{tag}.npy", oof); np.save(f"test_{tag}.npy", tp)
    oofs.append(oof); tests.append(tp); weights.append(w)

In [ ]:
# Ensemble: trộn xác suất theo trọng số, rồi tinh chỉnh ngưỡng lớp hiếm trên OOF
w = np.array(weights)/sum(weights)
oof_ens = sum(wi*o for wi,o in zip(w, oofs))
test_ens = sum(wi*t for wi,t in zip(w, tests))

from sklearn.metrics import f1_score, classification_report
print("OOF Macro F1 từng model:", [round(f1_score(y_all,o.argmax(1),average='macro'),4) for o in oofs])
print("OOF Macro F1 ENSEMBLE (argmax):", round(f1_score(y_all, oof_ens.argmax(1), average='macro'),4))

best_m, best = (1.0,1.0), -1
for wj in np.arange(0.8,1.31,0.05):
    for wp in np.arange(0.8,2.01,0.05):
        f = f1_score(y_all,(oof_ens*[1.0,wj,wp]).argmax(1),average="macro")
        if f>best: best,best_m = f,(round(float(wj),2),round(float(wp),2))
print(f"He so JAILBREAK x{best_m[0]}, PROMPT_INJECTION x{best_m[1]} -> OOF Macro F1 = {best:.4f}")
print(classification_report(y_all,(oof_ens*[1.0,*best_m]).argmax(1),target_names=LABELS,digits=4))

In [ ]:
# Xuất submission + kiểm tra toàn vẹn
pred = (test_ens*[1.0,*best_m]).argmax(1)
sub = pd.DataFrame({"id": test["id"], "label":[LABELS[i] for i in pred]})
assert len(sub)==6000 and sub["label"].isin(LABELS).all() and sub["label"].notna().all() and sub["id"].is_unique
sub.to_csv("submission.csv", index=False)
np.save("ensemble_test_proba.npy", test_ens)
print(sub["label"].value_counts().to_dict())

In [ ]:
try:
    from google.colab import files
    files.download("submission.csv"); files.download("ensemble_test_proba.npy")
except Exception: print("File o thu muc hien tai.")